<a href="https://colab.research.google.com/github/William-franklyn/gpu-diagnostics-test-suite/blob/main/training/gemma_lora_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import json, random
random.seed(7)

GPUS = [("Tesla T4", False, "15360 MiB"), ("NVIDIA A100-SXM4-40GB", False, "40960 MiB"),
        ("NVIDIA L4", False, "23034 MiB"), ("NVIDIA GeForce RTX 3090", True, "24576 MiB"),
        ("NVIDIA GeForce RTX 4090", True, "24564 MiB"), ("NVIDIA RTX A4000", True, "16376 MiB")]
DRIVERS = ["535.104.05", "550.54.15", "570.86.10", "580.82.07"]
SYSTEM = "You are a GPU diagnostics assistant. Read the test results and explain in simple words what happened, why, and what to do."

def make_example():
    name, has_fan, mem = random.choice(GPUS)
    fault = random.random() < 0.3
    lines, notes, fails = [], [], 0

    # temperature
    if fault:
        t = random.randint(35, 70)
        lines += [f"temperature: {t} C (limit 10 C)", "ERROR: GPU is too hot", "[temp] FAIL"]; fails += 1
        notes.append(f"Temperature failed on purpose. Fault injection lowered the limit to 10 C, and the GPU is really at {t} C, which is normal. This proves the temperature check works.")
    elif random.random() < 0.15:
        t = random.randint(85, 96)
        lines += [f"temperature: {t} C (limit 85 C)", "ERROR: GPU is too hot", "[temp] FAIL"]; fails += 1
        tip = "Check that the fans spin and the heatsink is not dusty." if has_fan else "This GPU has no fan of its own, so check the server's fans and airflow."
        notes.append(f"Temperature failed. The GPU is at {t} C, at or above the 85 C limit. It may slow itself down to cool off. {tip}")
    else:
        t = random.randint(30, 80)
        lines += [f"temperature: {t} C (limit 85 C)", "[temp] PASS"]
        notes.append(f"Temperature passed. {t} C is a safe level.")

    # fan
    if fault:
        lines += ["fan speed: 150%", "ERROR: fan speed is impossible", "[fan] FAIL"]; fails += 1
        notes.append("Fan failed on purpose. Fault injection set the speed to 150%, which is impossible. This proves the fan check catches bad readings.")
    elif not has_fan:
        lines += ["ERROR: could not read fan speed (this GPU may have no fan)", "[fan] FAIL"]; fails += 1
        notes.append(f"Fan failed, but this is expected. The {name} reports no fan, because it is cooled by the server. The hardware is fine.")
    elif random.random() < 0.1:
        lines += ["ERROR: could not read fan speed (this GPU may have no fan)", "[fan] FAIL"]; fails += 1
        notes.append(f"Fan failed. The {name} normally has a fan, so not being able to read it is a real warning. Check the fan cable and the driver.")
    else:
        s = random.randint(25, 90)
        lines += [f"fan speed: {s}%", "[fan] PASS"]
        notes.append(f"Fan passed. {s}% is a normal speed.")

    # memory
    if fault:
        lines += ["memory: 4 wrong values", "ERROR: GPU memory is not storing values correctly", "[memory] FAIL"]; fails += 1
        notes.append("Memory failed on purpose. Fault injection changed one value in each of the 4 patterns, so 4 values are wrong. This proves the memory check works.")
    elif random.random() < 0.1:
        k = random.choice([1, 3, 17, 250, 4096])
        lines += [f"memory: {k} wrong values", "ERROR: GPU memory is not storing values correctly", "[memory] FAIL"]; fails += 1
        notes.append(f"Memory failed. {k} values did not read back the same as they were written. This is a real hardware warning. Run the test again. If it keeps failing, the GPU memory may be damaged.")
    else:
        lines += ["memory: 0 wrong values", "[memory] PASS"]
        notes.append("Memory passed. Every value read back correctly.")

    # compute
    if fault:
        lines += ["compute: 1 wrong answers", "ERROR: GPU gave wrong answers", "[compute] FAIL"]; fails += 1
        notes.append("Compute failed on purpose. Fault injection changed one answer. This proves the compute check works.")
    elif random.random() < 0.08:
        k = random.choice([1, 12, 900])
        lines += [f"compute: {k} wrong answers", "ERROR: GPU gave wrong answers", "[compute] FAIL"]; fails += 1
        notes.append(f"Compute failed. The GPU got {k} additions wrong. That should never happen. Check if the GPU is overheating, then test again.")
    else:
        lines += ["compute: 0 wrong answers", "[compute] PASS"]
        notes.append("Compute passed. All answers were correct.")

    fan_line = "Fan: yes (card has its own fan)" if has_fan else "Fan: none (GPU reports no fan)"
    mode = "fault injection" if fault else "normal"
    user = (f"GPU: {name}\nMemory: {mem}\nDriver: {random.choice(DRIVERS)}\n{fan_line}\nMode: {mode}\n\n"
            + "\n".join("   " + l for l in lines) + f"\n\n   Summary: 4 run, {fails} failed")

    if fault:
        head = "This was a fault injection run, so the failures are expected. They show every check works."
    elif fails == 0:
        head = "All checks passed. This GPU looks healthy."
    else:
        head = f"{fails} of 4 checks failed. Here is what each result means."
    answer = head + "\n\n" + "\n".join("- " + n for n in notes)

    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": user},
                         {"role": "assistant", "content": answer}]}

data = [make_example() for _ in range(400)]
with open("gpu_diag_train.jsonl", "w") as f:
    for d in data[:360]:
        f.write(json.dumps(d) + "\n")
with open("gpu_diag_test.jsonl", "w") as f:
    for d in data[360:]:
        f.write(json.dumps(d) + "\n")

print("Made 360 training and 40 test examples.")
print(data[0]["messages"][1]["content"])

Made 360 training and 40 test examples.
GPU: NVIDIA L4
Memory: 23034 MiB
Driver: 535.104.05
Fan: none (GPU reports no fan)
Mode: normal

   temperature: 33 C (limit 85 C)
   [temp] PASS
   ERROR: could not read fan speed (this GPU may have no fan)
   [fan] FAIL
   memory: 4096 wrong values
   ERROR: GPU memory is not storing values correctly
   [memory] FAIL
   compute: 0 wrong answers
   [compute] PASS

   Summary: 4 run, 2 failed


In [11]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [42]:
#MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_NAME = "google/gemma-2-2b-it"

In [43]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [44]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "gpu_diag_train.jsonl",
                                      "test": "gpu_diag_test.jsonl"})

def fix(example):
    msgs = example["messages"]
    # Gemma doesn't accept a "system" message, so join it into the user message
    if "gemma" in MODEL_NAME.lower():
        msgs = [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]},
                msgs[2]]
    return {"messages": msgs}

ds = ds.map(fix)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 360
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 40
    })
})


In [50]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                             quantization_config=bnb,
                                             device_map="auto",
                                             torch_dtype=torch.float32,
                                             attn_implementation="eager")

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [51]:
def ask(model, example):
    msgs = example["messages"][:-1]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

sample = ds["test"][0]
print(sample["messages"][-2]["content"])
before = ask(model, sample)
print("\nBEFORE:\n", before)

You are a GPU diagnostics assistant. Read the test results and explain in simple words what happened, why, and what to do.

GPU: NVIDIA L4
Memory: 23034 MiB
Driver: 550.54.15
Fan: none (GPU reports no fan)
Mode: normal

   temperature: 47 C (limit 85 C)
   [temp] PASS
   ERROR: could not read fan speed (this GPU may have no fan)
   [fan] FAIL
   memory: 0 wrong values
   [memory] PASS
   compute: 0 wrong answers
   [compute] PASS

   Summary: 4 run, 1 failed

BEFORE:
 Okay, let's break down these GPU diagnostics results.

**What happened:**

* **Temperature:** Your GPU's temperature is within a safe range. It's currently at 47°C, which is below the maximum of 85°C. This is good news!
* **Fan:** The test couldn't determine if your GPU has a fan.  Since it's reporting "no fan," it's possible your GPU doesn't have one.
* **Memory:** The memory test passed, meaning your GPU's memory is working correctly.
* **Compute:** The compute test passed, meaning your GPU is performing calculations co

In [53]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules="all-linear", task_type="CAUSAL_LM")

args = SFTConfig(output_dir="lora-out",
                 num_train_epochs=2,
                 per_device_train_batch_size=2,
                 gradient_accumulation_steps=4,
                 learning_rate=2e-4,
                 logging_steps=10,
                 fp16=False,
                 bf16=False,
                 max_length=1024,
                 save_strategy="no",
                 report_to="none")

trainer = SFTTrainer(model=model, args=args,
                     train_dataset=ds["train"],
                     peft_config=lora,
                     processing_class=tokenizer)
trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:342: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
10,1.153693
20,0.190018
30,0.101932
40,0.078608
50,0.054154
60,0.046980
70,0.043736
80,0.040320
90,0.041594


TrainOutput(global_step=90, training_loss=0.19455938239892323, metrics={'train_runtime': 711.4805, 'train_samples_per_second': 1.012, 'train_steps_per_second': 0.126, 'total_flos': 2681613555240960.0, 'train_loss': 0.19455938239892323, 'epoch': 2.0})

In [55]:
trained = trainer.model
trained.eval()
after = ask(trained, sample)

print("BEFORE:\n", before)
print("\nAFTER:\n", after)
print("\nRIGHT ANSWER:\n", sample["messages"][-1]["content"])

chip_name = MODEL_NAME.split("/")[-1] + "-gpu-diag-lora"
trained.save_pretrained(chip_name)
tokenizer.save_pretrained(chip_name)
print("Saved your LoRA chip in folder:", chip_name)

BEFORE:
 Okay, let's break down these GPU diagnostics results.

**What happened:**

* **Temperature:** Your GPU's temperature is within a safe range. It's currently at 47°C, which is below the maximum of 85°C. This is good news!
* **Fan:** The test couldn't determine if your GPU has a fan.  Since it's reporting "no fan," it's possible your GPU doesn't have one.
* **Memory:** The memory test passed, meaning your GPU's memory is working correctly.
* **Compute:** The compute test passed, meaning your GPU is performing calculations correctly.
* **Summary:**  You ran 4 tests, and 1 of them failed. This is a bit concerning, but we'll figure it out.

**Why it might have failed:**

* **Fan Issue:** The "fan" test failed because the test couldn't read the fan speed. This could be because your GPU doesn't have a fan, or the fan is not functioning properly. 

**What to do:**

1. **Check for a Fan:**  If you have a GPU, make sure it

AFTER:
 1 of 4 checks failed. Here is what each result means.

-

In [56]:
!zip -r gemma-chip.zip gemma-2-2b-it-gpu-diag-lora

from google.colab import files
files.download("gemma-chip.zip")

  adding: gemma-2-2b-it-gpu-diag-lora/ (stored 0%)
  adding: gemma-2-2b-it-gpu-diag-lora/tokenizer_config.json (deflated 50%)
  adding: gemma-2-2b-it-gpu-diag-lora/chat_template.jinja (deflated 52%)
  adding: gemma-2-2b-it-gpu-diag-lora/adapter_config.json (deflated 62%)
  adding: gemma-2-2b-it-gpu-diag-lora/adapter_model.safetensors (deflated 21%)
  adding: gemma-2-2b-it-gpu-diag-lora/README.md (deflated 65%)
  adding: gemma-2-2b-it-gpu-diag-lora/tokenizer.json (deflated 84%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!unzip -q -o gemma-chip.zip

from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_WRITE_TOKEN"))
username = api.whoami()["name"]
repo_id = username + "/gemma-2-2b-it-gpu-diag-lora"

api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="gemma-2-2b-it-gpu-diag-lora", repo_id=repo_id)

print("Uploaded! See it here: https://huggingface.co/" + repo_id)

Uploaded! See it here: https://huggingface.co/Willms/gemma-2-2b-it-gpu-diag-lora
